In [ ]:
#import argparse
import json
from pathlib import Path
import gymnasium as gym
import random
import argparse

from config_dqn import DQNConfig
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import os


In [ ]:
def make_env(env_id: str, seed: int) -> gym.Env:
    env = gym.make(env_id)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    return Monitor(env)

def test_hyperparameter(env_id: str =  'CartPole-v1',
                        seed: int | None = None, 
                        timesteps_per_round: int = 1000,
                        max_rounds: int = 2500,
                        reward_stop_count: int = 10,
                        min_stop_value: int = 499,
                        evaluation_episodes: int = 10,
                        **kwargs) -> dict:

    #timesteps_per_round: the number of training steps you learn per round.  You check the reward after each round to measure if the model is trained
    #max_rounds: Stop the experiment after this stage.  Usually use the number of rounds necessary to train the default.
    #reward_stop_count: This is to check how sustained your training is.  It requires your model to sustain the min_stop_value over several training rounds.
    #min_stop_value: This is the value for the reward that is your objective. The maximum for the cartpole environment is 500, meaning it is vertical for 500 steps.
    #evaluation_episodes: the evaluate_policy function tests the model over this number of episodes and returns the mean reward.
    #kwargs is a dictionary with the keys/values as the DQN hyperparameters
    
    #Environment    
    env = make_env(env_id, seed)

    #Construct model
    config = DQNConfig.from_dict(kwargs["kwargs"])
    tensorboard_log = None
    params = config.to_kwargs(seed=seed)
    model = DQN("MlpPolicy", env, **params)

    #Train the model until you reach your in terms of  
    rewardList = []
    round_number=0
    max_reward_count = 0
    while (round_number < max_rounds) and (max_reward_count < reward_stop_count):
        model.learn(total_timesteps=timesteps_per_round)
        mean_reward, std_reward = evaluate_policy(
            model,
            model.get_env(),
            n_eval_episodes=evaluation_episodes,
            deterministic=True,
        )
        if round_number % 100 == 0:
            print(round_number)
            print(mean_reward)
        rewardList.append(mean_reward)
        if mean_reward > min_stop_value:
            max_reward_count = max_reward_count + 1
        else:
            max_reward_count = 0
        round_number = round_number+1
    
    reward_dct = {"training_rounds": round_number,
                    "timestamps_per_round": timesteps_per_round,
                    "final_reward": rewardList[-1],
                    "reward_data": rewardList}
    
    return({**params, **reward_dct})

In [ ]:
#These are the differesnt values that we would like to check.
env_id = 'CartPole-v1'
varList = {"learning_rate": [2e-4,5e-5],
    "buffer_size": [2000000, 500000],
    "learning_starts": [90, 110],
    "batch_size": [25, 40],
    "tau": [.9, .95],
    "gamma": [.98, .97],
    "train_freq": [(2, "episode"),(10, "step")],
    "gradient_steps": [2,-1],
    "n_steps": [2,3],
    "target_update_interval": [9000, 11000],
    "exploration_fraction": [0.08, 0.12],
    "exploration_initial_eps": [0.95,0.99],
    "exploration_final_eps": [.045, .055],
    "max_grad_norm": [9,11]}

In [ ]:
'''1 dim grid search'''
resultList = []
max_rounds = 1500
reward_stop_count = 10
min_stop_value= 499
outputPath = "outputs/"

for var_key in varList:
    valList = varList[var_key]
    seed = random.randint(0,10000)
    for j in range(len(valList)):
        val = valList[j]
        filename = outputPath + 'trial_' + var_key + "_" + str(j) +'.json'
        param = {var_key: val}
        output = test_hyperparameter(env_id='CartPole-v1',
                                    seed=seed,
                                    max_rounds=max_rounds,
                                    kwargs=param)
        with open(filename, 'w') as f:
            json.dump(output, f)

In [ ]:
import random
'''RANDOM grid search'''
resultList = []
max_rounds = 10
outputPath = "outputs/"

trial_length = 25
for trial_num in range(trial_length):
    print("Trial " + str(trial_num))
    seed = random.randint(0,10000)
    
    #These are the differesnt random values that we would like to check.
    param = {"learning_rate": max(1e-4,random.gauss(2e-4,1e-4)),
        "buffer_size": max(0, int(random.gauss(2000000,500000))),
        "batch_size": max(10,int(random.gauss(32,6))),
        "tau": min(.995, max(0.9, int(random.gauss(0.95,.2)))),
        "gamma": min(.995, max(0.975, int(random.gauss(0.985,.005)))),
        "gradient_steps": [1,2,3,4,-1][random.randint(0,4)],
        "exploration_fraction": max(0.01,random.gauss(0.1,0.02)),
        "exploration_initial_eps": min(1.0,random.gauss(1.0,0.02))}

    #Write the results
    filename = outputPath + 'random_trial_'+ str(trial_num) +'.json'
    output =test_hyperparameter(env_id='CartPole-v1',
                                    seed=seed,
                                    max_rounds=max_rounds,
                                    kwargs=param)
    with open(filename, 'w') as f:
        json.dump(output, f)
